## Import

Set the project root and GPU visibility before importing `torch` or `vllm`. This notebook is meant for interactive debugging; production runs should still use `main.py` with `configs/BRIDGE.yaml`.


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "main.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root containing main.py")
os.chdir(PROJECT_ROOT)

# Respect an externally configured CUDA_VISIBLE_DEVICES; otherwise use this notebook default.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1,2,3")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "False")

list_gpu = [gpu.strip() for gpu in os.environ["CUDA_VISIBLE_DEVICES"].split(",") if gpu.strip()]
num_gpus = len(list_gpu)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("num_gpus:", num_gpus)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
from vllm import SamplingParams

from util.seed import seed_inference

In [ ]:
from dataset.process import format_chat
from dataset.task import get_cls_for_all_tasks, get_type_task
from util.runtime import parse_prompt_mode

## Args

Configure the model, prompt mode, and task here. Environment variables can override the notebook defaults without editing the file:

- `BENCHMARK_MODEL_NAME`
- `BENCHMARK_PROMPT_MODE`
- `BENCHMARK_TASK_NAME`
- `MODEL_FILE`
- `NUM_SAMPLE`

Supported prompt modes are `direct`, `cot`, `direct-N-shot`, and `cot-N-shot`.


In [ ]:
model_name = os.environ.get("BENCHMARK_MODEL_NAME", "Qwen3-30B-A3B-Instruct-2507")
print("model_name:", model_name)

In [ ]:
prompt_mode = os.environ.get("BENCHMARK_PROMPT_MODE", "direct")
prompt_base, num_example = parse_prompt_mode(prompt_mode)
print("prompt_mode:", prompt_mode)
print("prompt_base:", prompt_base)
print("num_example:", num_example)

In [ ]:
class EmptyArgs:
    def __init__(self):
        self.gpus = list_gpu
        self.engine = "vllm"
        self.model_name = model_name
        self.model_file = os.environ.get("MODEL_FILE", "configs/dict_model_path.json")
        self.seed = 42
        self.gpu_memory_util = 0.8
        self.max_token_all = 10 * 1024
        self.max_token_input = 8 * 1024
        self.max_token_output = self.max_token_all - self.max_token_input
        self.dir_dataset = ["dataset_raw"]
        self.dir_result = ["result"]
        self.dir_result_save = self.dir_result[0]


args = EmptyArgs()
if not args.gpus:
    raise ValueError("No GPU id found. Set CUDA_VISIBLE_DEVICES before loading the model.")
if not Path(args.model_file).exists():
    raise FileNotFoundError(f"Model mapping file does not exist: {args.model_file}")

args.seed = seed_inference(args.seed)
for k, v in vars(args).items():
    print(f"{k}: {v}")

## Model and Tokenizer Initialization

This cell loads the model into GPU memory. Run it only after confirming the GPU list, model name, token budget, and model mapping file above.


In [ ]:
from model.init import init_model

In [ ]:
# Initialize the model and tokenizer.
tokenizer, model = init_model(args)
model_name = args.model_name
print("runtime model_name:", model_name)
print("max_token_all:", args.max_token_all)
print("max_token_input:", args.max_token_input)
print("max_token_output:", args.max_token_output)

In [ ]:
sampling_params = SamplingParams(
    seed=args.seed,
    temperature=0.0,
    max_tokens=args.max_token_output,
    logprobs=-1,
)
print(sampling_params)

## Task class

The task data must follow the standard benchmark format: each row needs `id`, `split`, `language`, `type`, `instruction`, `input`, and `output`. Few-shot examples are loaded from `example/` or `example_cot/` according to `prompt_mode`.


In [ ]:
DICT_TYPE_TASK_CLS, DICT_TASK_CLS = get_cls_for_all_tasks()
print(f"Registered tasks: {len(DICT_TASK_CLS)}")
list(DICT_TASK_CLS.keys())[:10]

In [ ]:
task_name = os.environ.get("BENCHMARK_TASK_NAME", "BrainMRI-AIS")
if task_name not in DICT_TASK_CLS:
    raise KeyError(f"Task not registered: {task_name}")

In [ ]:
task = DICT_TASK_CLS[task_name](args=args, task=task_name)
task.setup(tokenizer=tokenizer, prompt_mode=prompt_mode)
print("Task:", task.name)
print("Task type:", get_type_task(task))
print("test size:", len(task.dataset_test))
print("examples:", len(task.examples))

## Inference - test one

Use one sample to inspect prompt formatting and model output before running a larger batch.


In [ ]:
if not task.dataset_test:
    raise ValueError(f"Task has no test samples: {task_name}")
idx = min(5, len(task.dataset_test) - 1)
data = task.dataset_test[idx]
data

In [ ]:
formatted_input = format_chat(
    model_name=model_name,
    tokenizer=tokenizer,
    data=data,
    max_token_input=args.max_token_input,
    examples=task.examples,
    chat_kwargs=getattr(args, "chat_kwargs", {}),
)
print(formatted_input)

In [ ]:
output = model.generate(formatted_input, sampling_params=sampling_params, use_tqdm=False)
for output_one in output:
    generated_text = output_one.outputs[0].text
    print(f"Prompt:\n\t{formatted_input}")
    print("-" * 50)
    print(f"Generated text:\n\t{generated_text}")

In [ ]:
completion = output_one.outputs[0]

In [ ]:
def _encode_without_special_tokens(tokenizer, text):
    try:
        return tokenizer.encode(text, add_special_tokens=False)
    except TypeError:
        return tokenizer(text, add_special_tokens=False)["input_ids"]


def _single_token_id(tokenizer, text):
    token_ids = _encode_without_special_tokens(tokenizer, text)
    if len(token_ids) != 1:
        raise ValueError(
            f"{text!r} is encoded into {len(token_ids)} tokens: {token_ids}. "
            "A single-position logit comparison needs each label variant to be one token."
        )
    return int(token_ids[0])


def _leading_whitespace(text):
    return text[: len(text) - len(text.lstrip())]


def show_label_token_candidates(tokenizer, labels=("Yes", "No"), prefixes=("", " ", "\n")):
    rows = []
    for label in labels:
        for prefix in prefixes:
            token_text = prefix + label
            token_ids = _encode_without_special_tokens(tokenizer, token_text)
            rows.append(
                {
                    "label": label,
                    "token_text": repr(token_text),
                    "token_ids": token_ids,
                    "is_single_token": len(token_ids) == 1,
                }
            )
    return pd.DataFrame(rows)


def find_label_token_position(completion, tokenizer, labels=("Yes", "No")):
    for pos, token_id in enumerate(completion.token_ids):
        token_text = tokenizer.decode([int(token_id)])
        if token_text.strip() in labels:
            return pos
    token_debug = [
        (pos, int(token_id), repr(tokenizer.decode([int(token_id)])))
        for pos, token_id in enumerate(completion.token_ids)
    ]
    raise ValueError(f"No standalone Yes/No token found in generated tokens: {token_debug}")


def get_yes_no_raw_logits(output_one, tokenizer, output_index=0):
    completion = output_one.outputs[output_index]
    if completion.logprobs is None:
        raise ValueError("completion.logprobs is None. Set SamplingParams(logprobs=-1 or top_n).")

    label_pos = find_label_token_position(completion, tokenizer)
    sampled_token_id = int(completion.token_ids[label_pos])
    sampled_token_text = tokenizer.decode([sampled_token_id])
    prefix = _leading_whitespace(sampled_token_text)

    yes_token_text = prefix + "Yes"
    no_token_text = prefix + "No"
    yes_token_id = _single_token_id(tokenizer, yes_token_text)
    no_token_id = _single_token_id(tokenizer, no_token_text)

    logits_at_pos = completion.logprobs[label_pos]
    missing = [
        (name, token_id)
        for name, token_id in (("Yes", yes_token_id), ("No", no_token_id))
        if token_id not in logits_at_pos
    ]
    if missing:
        raise ValueError(
            f"Missing label token ids in logprobs at position {label_pos}: {missing}. "
            "For whole-vocab lookup, initialize LLM with max_logprobs=-1 and use SamplingParams(logprobs=-1)."
        )

    rows = []
    for label, token_text, token_id in (
        ("Yes", yes_token_text, yes_token_id),
        ("No", no_token_text, no_token_id),
    ):
        item = logits_at_pos[token_id]
        rows.append(
            {
                "label": label,
                "token_text": repr(token_text),
                "token_id": token_id,
                "raw_logit": item.logprob,
                "rank": item.rank,
            }
        )

    df = pd.DataFrame(rows)
    yes_logit = float(df.loc[df["label"] == "Yes", "raw_logit"].iloc[0])
    no_logit = float(df.loc[df["label"] == "No", "raw_logit"].iloc[0])
    return {
        "generated_text": completion.text,
        "label_position": label_pos,
        "sampled_token_id": sampled_token_id,
        "sampled_token_text": sampled_token_text,
        "yes_logit": yes_logit,
        "no_logit": no_logit,
        "yes_minus_no_logit": yes_logit - no_logit,
        "table": df,
    }

In [ ]:
display(show_label_token_candidates(tokenizer))

In [ ]:
yes_no_logits_one = get_yes_no_raw_logits(output_one, tokenizer)
print("generated_text:", repr(yes_no_logits_one["generated_text"]))
print("label_position:", yes_no_logits_one["label_position"])
print("sampled_token:", yes_no_logits_one["sampled_token_id"], repr(yes_no_logits_one["sampled_token_text"]))
print("Yes - No raw logit margin:", yes_no_logits_one["yes_minus_no_logit"])

In [ ]:
display(yes_no_logits_one["table"])

## Inference - task

Run a bounded subset for interactive testing. Set `NUM_SAMPLE` to control the sample size without editing the notebook.


In [ ]:
num_sample = int(os.environ.get("NUM_SAMPLE", "100"))
if num_sample <= 0:
    raise ValueError("num_sample must be positive and the selected split must not be empty.")
print("num_sample:", num_sample)

In [ ]:
# Format the input text with the selected prompt mode.
list_input = []
list_num_token = []
for idx_data, dict_data in enumerate(task.dataset_test[:num_sample]):
    input_llm = format_chat(
        model_name=model_name,
        tokenizer=tokenizer,
        data=dict_data,
        max_token_input=args.max_token_input,
        examples=task.examples,
        chat_kwargs=getattr(args, "chat_kwargs", {}),
    )
    list_input.append(input_llm)
    len_token_input = len(tokenizer.encode(input_llm))
    list_num_token.append(len_token_input)
    if len_token_input > args.max_token_input:
        print(
            f"Input exceeds max token limit: idx={idx_data}, "
            f"tokens={len_token_input}, max={args.max_token_input}"
        )
print(f"Data size: {len(list_input)}")

In [ ]:
if len(list_input) > 1:
    print(list_input[1])
else:
    print(list_input[0])

In [ ]:
dict_stat_num_token = pd.Series(list_num_token).describe().to_dict()
print(dict_stat_num_token)

In [ ]:
# Count prompts that still exceed the configured input budget after formatting.
num_truncate = sum(num_token > args.max_token_input for num_token in list_num_token)
proportion_truncate = num_truncate / len(list_num_token) if list_num_token else 0.0
print(f"The number of data that may be truncated: {num_truncate}")
print(f"The proportion of data that may be truncated: {proportion_truncate:.2%}")

In [ ]:
plt.hist(list_num_token, bins=30, alpha=0.7)
plt.title('Token Count Distribution')
plt.xlabel('Number of Tokens')
plt.ylabel('Frequency')
plt.show()

In [ ]:
list_pred = []
output = model.generate(list_input, sampling_params=sampling_params, use_tqdm=True)
for output_one in output:
    generated_text = output_one.outputs[0].text
    list_pred.append(generated_text)

if len(list_pred) != len(list_input):
    raise ValueError(f"Prediction count mismatch: {len(list_pred)} vs {len(list_input)}")

In [ ]:
list_yes_no_logits = []
for idx_output, output_one in enumerate(output):
    item = get_yes_no_raw_logits(output_one, tokenizer)
    list_yes_no_logits.append(
        {
            "idx": idx_output,
            "generated_text": item["generated_text"],
            "label_position": item["label_position"],
            "sampled_token_id": item["sampled_token_id"],
            "sampled_token_text": repr(item["sampled_token_text"]),
            "yes_logit": item["yes_logit"],
            "no_logit": item["no_logit"],
            "yes_minus_no_logit": item["yes_minus_no_logit"],
        }
    )

df_yes_no_logits = pd.DataFrame(list_yes_no_logits)
display(df_yes_no_logits.head())

## Result

Attach predictions to the sampled task rows, parse them with the task-specific parser, and compute quick metrics. This section is for smoke testing; production result files should be written by `main.py`.


In [ ]:
for input_text, pred_text in zip(list_input, list_pred):
    print(f"Input text:\n\t{input_text}")
    print("-"*50)
    print(f"Generated text:\n\t{pred_text}")
    print("="*100)

In [ ]:
for idx_data, dict_data in enumerate(task.dataset_test[:num_sample]):
    dict_data["pred"] = list_pred[idx_data]
print(f"Attached predictions to {num_sample} samples.")

In [ ]:
list_eval_data = task.dataset_test[:num_sample]
list_pred_extracted = task.get_pred(list_eval_data, prompt_mode=prompt_mode)
list_label_extracted = task.get_label(list_eval_data, prompt_mode=prompt_mode)
print(f"Extracted {len(list_pred_extracted)} predictions and {len(list_label_extracted)} labels.")

### Performance

In [ ]:
list_pred_extracted, num_failed = task.get_pred_none(
    list_pred=list_pred_extracted[:num_sample],
    list_label=list_label_extracted[:num_sample],
)
print(f"The number of failed data: {num_failed} ({num_failed / num_sample:.2%})")

In [ ]:
task_type = get_type_task(task)
if task_type == "classification":
    dict_performance = task.get_performance(list_pred_extracted, list_label_extracted)
else:
    dict_performance, _ = task.get_performance(list_pred_extracted, list_label_extracted)
print("task_type:", task_type)


In [ ]:
dict_performance

## End

Release GPU memory by restarting the kernel when you finish interactive testing.


In [ ]:
print('Done!')